In [1]:
import pandas as pd
from pathlib import Path


In [8]:
from pathlib import Path

for name in ["USA", "Mexico"]:
    folder = Path(name)
    print("\nFolder:", folder.resolve())
    files = list(folder.glob("*.xls"))
    print("XLS encontrados:", len(files))
    if files:
        print("Ejemplo:", files[0].name)



Folder: C:\Users\mar_c\OneDrive\Documentos\GuruFocus\USA
XLS encontrados: 43
Ejemplo: ABG.xls

Folder: C:\Users\mar_c\OneDrive\Documentos\GuruFocus\Mexico
XLS encontrados: 34
Ejemplo: AC.xls


In [10]:
import pandas as pd
from pathlib import Path

def is_html_xls(path: Path) -> bool:
    """Muchos 'xls' de export web son HTML con extensión .xls."""
    try:
        with open(path, "rb") as f:
            head = f.read(2048).lower()
        return (b"<html" in head) or (b"<!doctype html" in head) or (b"<table" in head)
    except Exception:
        return False

def convert_html_xls_to_xlsx(xls_file: Path) -> bool:
    """Convierte HTML (guardado como .xls) a .xlsx leyendo tablas."""
    try:
        tables = pd.read_html(str(xls_file))  # lee todas las tablas del HTML
        if not tables:
            return False

        xlsx_path = xls_file.with_suffix(".xlsx")
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            for i, df in enumerate(tables, start=1):
                sheet = f"Table_{i}"
                df.to_excel(writer, sheet_name=sheet, index=False)

        return xlsx_path.exists()
    except Exception:
        return False

def convert_with_pandas_xlrd(xls_file: Path) -> bool:
    try:
        xls = pd.ExcelFile(xls_file, engine="xlrd")
        xlsx_path = xls_file.with_suffix(".xlsx")

        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            for sheet_name in xls.sheet_names:
                df = xls.parse(sheet_name)
                df.to_excel(writer, sheet_name=sheet_name, index=False)

        return xlsx_path.exists()
    except Exception:
        return False

def convert_with_excel_com(xls_file: Path) -> bool:
    """Usa Excel instalado. Importante: pasar ruta absoluta normalizada."""
    excel = None
    try:
        import win32com.client as win32

        abs_path = str(xls_file.resolve())  # <- CLAVE
        xlsx_path = xls_file.with_suffix(".xlsx")
        abs_xlsx = str(xlsx_path.resolve())

        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False

        wb = excel.Workbooks.Open(abs_path, CorruptLoad=1)
        wb.SaveAs(abs_xlsx, FileFormat=51)  # 51 = xlsx
        wb.Close(SaveChanges=False)
        excel.Quit()

        return Path(abs_xlsx).exists()
    except Exception:
        if excel is not None:
            try:
                excel.Quit()
            except Exception:
                pass
        return False

folders = [Path("USA"), Path("Mexico")]

ok_biff, ok_html, ok_excel, failed = [], [], [], []

for folder in folders:
    folder = folder.resolve()
    print("\n---", folder, "---")

    for xls_file in list(folder.glob("*.xls")) + list(folder.glob("*.XLS")):
        xls_file = xls_file.resolve()
        print(f"\nProcesando: {xls_file.name}")

        # 1) Si es HTML disfrazado de xls
        if is_html_xls(xls_file):
            if convert_html_xls_to_xlsx(xls_file):
                print("OK (HTML -> XLSX)")
                ok_html.append(str(xls_file))
                continue
            else:
                print("Fallo (HTML parse)")
                failed.append(str(xls_file))
                continue

        # 2) Intento BIFF real con xlrd
        if convert_with_pandas_xlrd(xls_file):
            print("OK (pandas/xlrd -> XLSX)")
            ok_biff.append(str(xls_file))
            continue

        # 3) Fallback con Excel
        if convert_with_excel_com(xls_file):
            print("OK (Excel COM -> XLSX)")
            ok_excel.append(str(xls_file))
            continue

        print("FALLÓ TODO")
        failed.append(str(xls_file))

print("\n=== RESUMEN ===")
print(f"Convertidos BIFF (pandas/xlrd): {len(ok_biff)}")
print(f"Convertidos HTML (read_html):   {len(ok_html)}")
print(f"Convertidos Excel COM:          {len(ok_excel)}")
print(f"Fallidos:                       {len(failed)}")

if failed:
    print("\nEjemplos fallidos (primeros 20):")
    for f in failed[:20]:
        print(" -", f)



--- C:\Users\mar_c\OneDrive\Documentos\GuruFocus\USA ---

Procesando: ABG.xls
_locate_stream(Workbook): seen
    0  5 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
   20  4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
 1360= 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
 1380  3 2 2 2 2 2 2 2 2 2 2 2 
OK (Excel COM -> XLSX)

Procesando: ABNB.xls
_locate_stream(Workbook): seen
    0  5 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
   20  4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
 1080= 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
 1100  4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 3 2 
 1120  2 2 2 2 2 2 2 2 
OK (Excel COM -> XLSX)

Procesando: ACN.xls
_locate_stream(Workbook): seen
    0  5 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
   20  4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
 1340= 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
 1360  4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 3 2 2 2 
 1380  2 2 2 2 2 2 2 2 
OK (Excel COM -> XLSX)

Procesando: ADBE.xls
_locate_stream(Workbook): seen
    0  5 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 
   20  

In [11]:
from pathlib import Path

folders = [Path("USA"), Path("Mexico")]

deleted = []
skipped = []

for folder in folders:
    folder = folder.resolve()
    print("\n---", folder, "---")

    for xls_file in list(folder.glob("*.xls")) + list(folder.glob("*.XLS")):
        xlsx_file = xls_file.with_suffix(".xlsx")

        if xlsx_file.exists():
            try:
                xls_file.unlink()
                print(f"BORRADO: {xls_file.name}")
                deleted.append(str(xls_file))
            except Exception as e:
                print(f"ERROR al borrar {xls_file.name}: {e}")
        else:
            skipped.append(str(xls_file))

print("\n=== RESUMEN ===")
print(f"Borrados: {len(deleted)}")
print(f"No borrados (sin xlsx): {len(skipped)}")

if skipped:
    print("\nEjemplos NO borrados (primeros 20):")
    for f in skipped[:20]:
        print(" -", f)



--- C:\Users\mar_c\OneDrive\Documentos\GuruFocus\USA ---
BORRADO: ABG.xls
BORRADO: ABNB.xls
ERROR al borrar ACN.xls: [WinError 32] El proceso no tiene acceso al archivo porque está siendo utilizado por otro proceso: 'C:\\Users\\mar_c\\OneDrive\\Documentos\\GuruFocus\\USA\\ACN.xls'
BORRADO: ADBE.xls
BORRADO: ADP.xls
BORRADO: ADSK.xls
BORRADO: AMD.xls
BORRADO: AMGN.xls
BORRADO: AMZN.xls
BORRADO: BAC.xls
BORRADO: BRO.xls
BORRADO: BX.xls
BORRADO: CDNS.xls
BORRADO: CI.xls
BORRADO: CMCSA.xls
BORRADO: CRM.xls
BORRADO: DECK.xls
BORRADO: FI.xls
BORRADO: FISV.xls
BORRADO: FTNT.xls
BORRADO: GOOG.xls
BORRADO: HD.xls
BORRADO: INTU.xls
BORRADO: LMT.xls
BORRADO: MA.xls
BORRADO: MELI.xls
BORRADO: META.xls
BORRADO: MMC.xls
BORRADO: MNST.xls
BORRADO: MORN.xls
BORRADO: MSFT.xls
BORRADO: NFLX.xls
BORRADO: NIKE.xls
BORRADO: PCAR.xls
BORRADO: PEP.xls
BORRADO: ROP.xls
BORRADO: TER.xls
BORRADO: TMUS.xls
BORRADO: TSM.xls
BORRADO: TTWO.xls
BORRADO: V.xls
BORRADO: VRTX.xls
BORRADO: WFC.xls
ERROR al borrar ABG.x